# Webull HK Data Probe

This notebook loads Webull OpenAPI credentials from `.env`, probes whether the account can fetch historical bars for HK ETF symbols such as `2800.HK`, and saves any successful result into `data/raw/` using the project schema:

`timestamp, open, high, low, close, volume`

Note: Webull's current public Market Data API docs list `US_STOCK` and `US_ETF` categories. If HK symbols fail, use Bloomberg/Futu exports for this project.

In [6]:
# Run once if the SDK is not installed in this notebook kernel.
%pip install --upgrade webull-openapi-python-sdk pandas


Note: you may need to restart the kernel to use updated packages.


In [7]:
import logging
import os
from pathlib import Path

import pandas as pd

logging.disable(logging.CRITICAL)

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    # If Jupyter starts inside notebooks/, move to repo root.
    ROOT = ROOT.parent


def load_env(path: Path = ROOT / ".env") -> None:
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key.strip(), value)


load_env()

required = ["WEBULL_APP_KEY", "WEBULL_APP_SECRET", "WEBULL_ACCOUNT_ID"]
missing = [key for key in required if not os.environ.get(key)]
if missing:
    raise RuntimeError(f"Missing required .env keys: {missing}")

print("Loaded Webull credentials from .env without displaying secrets.")
print("Repo root:", ROOT)


Loaded Webull credentials from .env without displaying secrets.
Repo root: /Users/yela/Documents/University Programming/RL_quantphemes_2


In [8]:
from urllib.parse import urlparse

from webull.core.client import ApiClient
from webull.data.common.category import Category
from webull.data.common.timespan import Timespan
from webull.data.data_client import DataClient

app_key = os.environ["WEBULL_APP_KEY"]
app_secret = os.environ["WEBULL_APP_SECRET"]
preferred_region = os.environ.get("WEBULL_REGION", "hk").lower()
if preferred_region not in {"hk", "us"}:
    print(f"Ignoring invalid WEBULL_REGION={preferred_region!r}; using 'hk'.")
    preferred_region = "hk"
preferred_endpoint = os.environ.get("WEBULL_API_ENDPOINT", "").strip()


def _normalize_endpoint(endpoint: str) -> str:
    endpoint = endpoint.strip().rstrip("/")
    if not endpoint:
        return ""
    parsed = urlparse(endpoint)
    if parsed.netloc:
        return parsed.netloc
    return endpoint


preferred_endpoint = _normalize_endpoint(preferred_endpoint)


def _dedupe(items: list[tuple[str, str | None]]) -> list[tuple[str, str | None]]:
    seen = set()
    out = []
    for item in items:
        if item in seen:
            continue
        seen.add(item)
        out.append(item)
    return out


endpoint_candidates = _dedupe(
    [
        (preferred_region, preferred_endpoint or None),
        ("hk", None),
        ("hk", "api.webull.hk"),
        ("hk", "api.sandbox.webull.hk"),
        ("hk", "hk-openapi.uat.webullbroker.com"),
        ("us", None),
        ("us", "api.webull.com"),
    ]
)

auth_attempts = []
data_client = None
api_client = None
active_region = None
active_endpoint = None

for candidate_region, candidate_endpoint in endpoint_candidates:
    try:
        client = ApiClient(app_key, app_secret, candidate_region)
        if candidate_endpoint:
            client.add_endpoint(candidate_region, candidate_endpoint)
        candidate_data_client = DataClient(client)
    except Exception as exc:
        auth_attempts.append(
            {
                "region": candidate_region,
                "endpoint": candidate_endpoint or "sdk-default",
                "ok": False,
                "error": repr(exc)[:240],
            }
        )
        continue

    api_client = client
    data_client = candidate_data_client
    active_region = candidate_region
    active_endpoint = candidate_endpoint or "sdk-default"
    auth_attempts.append(
        {
            "region": candidate_region,
            "endpoint": active_endpoint,
            "ok": True,
            "error": "",
        }
    )
    break

auth_attempts_df = pd.DataFrame(auth_attempts)
display(auth_attempts_df)

if data_client is None:
    raise RuntimeError(
        "Webull SDK authentication failed for every region/endpoint candidate. "
        "Check that the key/secret are OpenAPI keys, the token is NORMAL/enabled, "
        "and WEBULL_REGION matches the application region, usually 'hk' for Webull HK."
    )

print("SDK client initialized.")
print("Active region:", active_region)
print("Active endpoint:", active_endpoint)
print("Available Category members:", [item.name for item in Category])
print("Available Timespan members:", [item.name for item in Timespan])


,region,endpoint,ok,error
0,hk_etf,api.sandbox.webull.hk,True,


SDK client initialized.
Active region: hk_etf
Active endpoint: api.sandbox.webull.hk
Available Category members: ['US_STOCK', 'US_OPTION', 'HK_STOCK', 'US_ETF', 'HK_ETF', 'CN_STOCK', 'US_CRYPTO', 'US_FUTURES', 'US_EVENT', 'HK_FUTURES']
Available Timespan members: ['S5', 'S15', 'M1', 'M5', 'M15', 'M30', 'M60', 'M120', 'M240', 'D', 'W', 'M', 'Y']


In [9]:
# Probe only sensible Webull category/symbol combinations.
# 2800.HK is Tracker Fund of Hong Kong, so HK_ETF is the primary category.
probe_plan = [
    {"symbol": "2800", "category": Category.HK_ETF.name},
    {"symbol": "02800", "category": Category.HK_ETF.name},
    {"symbol": "HK.2800", "category": Category.HK_ETF.name},
    {"symbol": "2800.HK", "category": Category.HK_ETF.name},
    {"symbol": "2800", "category": Category.HK_STOCK.name},
    {"symbol": "02800", "category": Category.HK_STOCK.name},
    # Sandbox/test-account sanity check from Webull docs.
    {"symbol": "AAPL", "category": Category.US_STOCK.name},
]
timespans = [Timespan.M1.name, Timespan.M5.name, Timespan.M30.name, Timespan.D.name]


def _response_payload(response: object) -> object | None:
    status = getattr(response, "status_code", None)
    if status != 200:
        return None
    return response.json()


def _error_summary(exc: Exception) -> str:
    text = str(exc) or repr(exc)
    request_id = getattr(exc, "request_id", None)
    code = getattr(exc, "code", None) or getattr(exc, "error_code", None)
    bits = []
    if code:
        bits.append(f"code={code}")
    if request_id:
        bits.append(f"request_id={request_id}")
    bits.append(text[:240])
    return " | ".join(bits)


probe_results = []
for item in probe_plan:
    for timespan in timespans:
        symbol = item["symbol"]
        category = item["category"]
        try:
            response = data_client.market_data.get_history_bar(
                symbol,
                category,
                timespan,
                count="20",
            )
            status = getattr(response, "status_code", None)
            payload = _response_payload(response)
            size = len(payload) if isinstance(payload, list) else None
            ok = status == 200 and bool(payload)
            probe_results.append(
                {
                    "symbol": symbol,
                    "category": category,
                    "timespan": timespan,
                    "status": status,
                    "size": size,
                    "ok": ok,
                    "error": "",
                    "payload": payload,
                }
            )
            print(symbol, category, timespan, "status=", status, "size=", size)
        except Exception as exc:
            error = _error_summary(exc)
            probe_results.append(
                {
                    "symbol": symbol,
                    "category": category,
                    "timespan": timespan,
                    "status": "exception",
                    "size": None,
                    "ok": False,
                    "error": error,
                    "payload": None,
                }
            )
            print(symbol, category, timespan, "ERROR", error)

summary = pd.DataFrame(
    [{k: v for k, v in row.items() if k != "payload"} for row in probe_results]
)
summary


2800 HK_ETF M1 ERROR code=INVALID_SYMBOL | request_id=30563789-a339-4b40-8d15-43b7ed83aeb3 | HTTP Status: 417, Code: INVALID_SYMBOL, Msg: The symbol does not exist in the category., RequestID: 30563789-a339-4b40-8d15-43b7ed83aeb3
2800 HK_ETF M5 ERROR code=INVALID_SYMBOL | request_id=2a3e8f8a-877a-4e6a-9deb-05a0d500487a | HTTP Status: 417, Code: INVALID_SYMBOL, Msg: The symbol does not exist in the category., RequestID: 2a3e8f8a-877a-4e6a-9deb-05a0d500487a
2800 HK_ETF M30 ERROR code=INVALID_SYMBOL | request_id=5339f078-85b0-4779-8754-5dcf252397c1 | HTTP Status: 417, Code: INVALID_SYMBOL, Msg: The symbol does not exist in the category., RequestID: 5339f078-85b0-4779-8754-5dcf252397c1
2800 HK_ETF D ERROR code=INVALID_SYMBOL | request_id=756fd9f5-798a-4b08-9496-c2cc31403b23 | HTTP Status: 417, Code: INVALID_SYMBOL, Msg: The symbol does not exist in the category., RequestID: 756fd9f5-798a-4b08-9496-c2cc31403b23
02800 HK_ETF M1 status= 200 size= 20
02800 HK_ETF M5 status= 200 size= 20
02800 

,symbol,category,timespan,status,size,ok,error
0,2800,HK_ETF,M1,exception,NaN,False,code=INVALID_SYMBOL | request_id=30563789-a339...
1,2800,HK_ETF,M5,exception,NaN,False,code=INVALID_SYMBOL | request_id=2a3e8f8a-877a...
2,2800,HK_ETF,M30,exception,NaN,False,code=INVALID_SYMBOL | request_id=5339f078-85b0...
3,2800,HK_ETF,D,exception,NaN,False,code=INVALID_SYMBOL | request_id=756fd9f5-798a...
4,02800,HK_ETF,M1,200,20.0,True,
5,02800,HK_ETF,M5,200,20.0,True,
6,02800,HK_ETF,M30,200,20.0,True,
7,02800,HK_ETF,D,200,20.0,True,
8,HK.2800,HK_ETF,M1,exception,NaN,False,code=INVALID_SYMBOL | request_id=4ed3151c-bc81...
9,HK.2800,HK_ETF,M5,exception,NaN,False,code=INVALID_SYMBOL | request_id=70dbfc33-0f68...


In [10]:
def normalize_webull_bars(payload: object) -> pd.DataFrame:
    """Normalize likely Webull bar payloads to timestamp/open/high/low/close/volume."""
    if isinstance(payload, dict):
        for key in ("data", "bars", "items"):
            if key in payload:
                payload = payload[key]
                break
    if not isinstance(payload, list) or not payload:
        return pd.DataFrame(columns=["timestamp", "open", "high", "low", "close", "volume"])

    frame = pd.DataFrame(payload)
    rename = {}
    for source, target in {
        "time": "timestamp",
        "date": "timestamp",
        "datetime": "timestamp",
        "t": "timestamp",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
    }.items():
        if source in frame.columns and target not in frame.columns:
            rename[source] = target
    frame = frame.rename(columns=rename)

    if "timestamp" not in frame.columns:
        columns = frame.columns.tolist()
        raise ValueError(f"Could not identify timestamp column. Columns: {columns}")

    # Webull docs say timestamps may be Unix milliseconds. Also handle ISO strings.
    if pd.api.types.is_numeric_dtype(frame["timestamp"]):
        frame["timestamp"] = pd.to_datetime(
            frame["timestamp"],
            unit="ms",
            utc=True,
        ).dt.tz_convert("Asia/Hong_Kong")
    else:
        frame["timestamp"] = pd.to_datetime(
            frame["timestamp"],
            utc=True,
            errors="coerce",
        ).dt.tz_convert("Asia/Hong_Kong")
    frame["timestamp"] = frame["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")

    for col in ["open", "high", "low", "close", "volume"]:
        if col not in frame.columns:
            raise ValueError(f"Missing {col!r}. Columns: {frame.columns.tolist()}")
        frame[col] = pd.to_numeric(frame[col], errors="coerce")

    columns = ["timestamp", "open", "high", "low", "close", "volume"]
    return frame[columns].dropna().sort_values("timestamp")


successful = [row for row in probe_results if row.get("ok")]
print("Successful probes:", len(successful))
for row in successful[:5]:
    print(row["symbol"], row["category"], row["timespan"], "bars", row["size"])

successful[:1]


Successful probes: 12
02800 HK_ETF M1 bars 20
02800 HK_ETF M5 bars 20
02800 HK_ETF M30 bars 20
02800 HK_ETF D bars 20
02800 HK_STOCK M1 bars 20


[{'symbol': '02800',
  'category': 'HK_ETF',
  'timespan': 'M1',
  'status': 200,
  'size': 20,
  'ok': True,
  'error': '',
  'payload': [{'tickerId': '925186414',
    'symbol': '02800',
    'time': '2026-05-13T07:59:00.000+0000',
    'open': '26.54',
    'close': '26.52',
    'high': '26.54',
    'low': '26.52',
    'volume': '1252000',
    'trading_session': 'RTH'},
   {'tickerId': '925186414',
    'symbol': '02800',
    'time': '2026-05-13T07:58:00.000+0000',
    'open': '26.52',
    'close': '26.54',
    'high': '26.54',
    'low': '26.52',
    'volume': '1260525',
    'trading_session': 'RTH'},
   {'tickerId': '925186414',
    'symbol': '02800',
    'time': '2026-05-13T07:57:00.000+0000',
    'open': '26.54',
    'close': '26.52',
    'high': '26.54',
    'low': '26.52',
    'volume': '1624500',
    'trading_session': 'RTH'},
   {'tickerId': '925186414',
    'symbol': '02800',
    'time': '2026-05-13T07:56:00.000+0000',
    'open': '26.52',
    'close': '26.54',
    'high': '26.5

In [11]:
# Save the first successful HK 2800 response. If only AAPL succeeds, that confirms sandbox limits.
successful_2800 = [
    row
    for row in successful
    if row["symbol"] in {"2800", "02800", "HK.2800", "2800.HK"}
]

if not successful_2800:
    successful_symbols = sorted({row["symbol"] for row in successful})
    print(
        "No successful HK 2800 historical bars returned. "
        f"Successful symbols in this run: {successful_symbols}. "
        "If AAPL succeeds on sandbox but 2800 does not, the SDK is working and "
        "the remaining issue is sandbox market coverage or production data entitlement."
    )
else:
    selected = successful_2800[0]
    bars = normalize_webull_bars(selected["payload"])
    safe_symbol = selected["symbol"].replace(".", "_").replace("/", "_")
    out = ROOT / "data" / "raw" / f"webull_{safe_symbol}_{selected['timespan'].lower()}.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    bars.to_csv(out, index=False)
    print("Saved", len(bars), "bars to", out)
    display(bars.head())
    display(bars.tail())


Saved 20 bars to /Users/yela/Documents/University Programming/RL_quantphemes_2/data/raw/webull_02800_m1.csv


,timestamp,open,high,low,close,volume
19,2026-05-13 15:40:00,26.54,26.54,26.52,26.54,143000
18,2026-05-13 15:41:00,26.52,26.54,26.52,26.54,169120
17,2026-05-13 15:42:00,26.54,26.54,26.54,26.54,147058
16,2026-05-13 15:43:00,26.54,26.54,26.54,26.54,143000
15,2026-05-13 15:44:00,26.54,26.54,26.52,26.52,1493000


,timestamp,open,high,low,close,volume
4,2026-05-13 15:55:00,26.52,26.52,26.50,26.52,75520
3,2026-05-13 15:56:00,26.52,26.54,26.52,26.54,3478000
2,2026-05-13 15:57:00,26.54,26.54,26.52,26.52,1624500
1,2026-05-13 15:58:00,26.52,26.54,26.52,26.54,1260525
0,2026-05-13 15:59:00,26.54,26.54,26.52,26.52,1252000
